In [22]:
!pip install openai
!pip install groq


In [23]:
import os
from openai import OpenAI
from groq import Groq

In [24]:

# Set your Groq API Key
groq_api_key = "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"
os.environ["GROQ_API_KEY"] = groq_api_key

# Create client
client = Groq(api_key=groq_api_key)

def prompt_based_query(user_query):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",  # or another Groq-supported model
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": user_query}
        ],
        temperature=0.2
    )

    return response.choices[0].message.content

# Example
answer = prompt_based_query("What is Machine Learning?")
print(answer)

**Machine Learning (ML)** is a subset of Artificial Intelligence (AI) that involves training algorithms to learn from data and make predictions or decisions without being explicitly programmed. It enables computers to automatically improve their performance on a task by learning from experience, rather than relying on manual programming.

**Key Characteristics of Machine Learning:**

1. **Data-Driven**: ML algorithms learn from data, which can be in the form of images, text, audio, or other types of information.
2. **Pattern Recognition**: ML algorithms identify patterns in the data, such as relationships between variables or anomalies.
3. **Prediction**: ML algorithms use the learned patterns to make predictions or decisions about new, unseen data.
4. **Improvement over Time**: ML algorithms can improve their performance over time as they receive more data and learn from their mistakes.

**Types of Machine Learning:**

1. **Supervised Learning**: The algorithm is trained on labeled da

In [25]:
!pip install langchain faiss-cpu openai langchain_community tiktoken

In [26]:
from langchain_community.document_loaders import TextLoader

# Load the text file
loader = TextLoader("/content/medical_company_data.txt", encoding="utf-8")

# Read the document
documents = loader.load()

# Print content
print(documents[0].page_content)

Company Name: MediCare Plus Pvt Ltd

About:
MediCare Plus is a digital healthcare platform offering online doctor consultations, pharmacy delivery, lab test bookings, and health record management. We connect patients with certified doctors across general medicine, pediatrics, dermatology, cardiology, and more.

Services:

Online Doctor Consultation:
Patients can book video, audio, or chat consultations with certified doctors.
Consultations are available for General Medicine, Pediatrics, Dermatology, Cardiology, Orthopedics, Gynecology, and Mental Health.
Consultation fees vary by specialty and doctor experience.

Appointment Policy:
Appointments can be booked online via the website or mobile app.
Patients must book at least 30 minutes in advance for same-day consultations.
Appointments can be rescheduled or cancelled up to 15 minutes before the scheduled time.
A full refund is issued for cancellations made at least 1 hour before the appointment.

Prescription Policy:
Digital prescripti

In [27]:
!pip install -q langchain-huggingface sentence-transformers faiss-cpu

In [28]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_db = FAISS.from_documents(documents, embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [29]:
from groq import Groq

client = Groq(api_key='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')

In [30]:
def rag_query(query):
    # Retrieve relevant chunks
    docs = vector_db.similarity_search(query, k=3)

    # Build context
    context = "\n\n".join(doc.page_content for doc in docs)

    # Generate answer using Groq
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": "Use only the provided context to answer the question. If the answer is not in the context, say 'I don't know based on the provided document.'"
            },
            {
                "role": "user",
                "content": f"Context:\n{context}\n\nQuestion: {query}"
            }
        ],
        temperature=0.2
    )

    return response.choices[0].message.content

In [31]:
print(rag_query("What is our company Prescription policy?"))

Our company's Prescription Policy is as follows:

1. Digital prescriptions are issued by doctors after every consultation.
2. Prescriptions are valid for 30 days from the date of issue.
3. Prescriptions are accessible anytime from the patient's health dashboard.
4. MediCare Plus does not issue prescriptions for controlled or narcotic substances online.


In [32]:
!pip install transformers datasets bitsandbytes peft accelerate

In [33]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import bitsandbytes as bnb
model_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer=AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token # Set pad_token for causal language modeling
#define the bitsandbytes config for 8 bit quantization
quantization_config=BitsAndBytesConfig(load_in_8bit=True)
#load the model
model=AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto"
    )

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [34]:
from peft import LoraConfig,get_peft_model,prepare_model_for_kbit_training
model= prepare_model_for_kbit_training(model)
lora_config=LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.5,
    bias="none",
    task_type="CAUSAL_LM"
)
model=get_peft_model(model,lora_config)
# Enable input gradients for the PeftModel (sometimes needed explicitly for Trainer compatibility)
model.enable_input_require_grads()
model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 2048)
        (layers): ModuleList(
          (0-21): 22 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear8bitLt(
                (base_layer): Linear8bitLt(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.5, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Lin

In [35]:
from datasets import Dataset
data=[
    {"text":"Q:What is Appointment Policy?\nA:Appointments can be booked online via the website or mobile app,Patients must book at least 30 minutes in advance for same-day consultations.Appointments can be rescheduled or cancelled up to 15 minutes before the scheduled time.A full refund is issued for cancellations made at least 1 hour before the appointment."}
]
dataset = Dataset.from_list(data)

In [36]:
#Step-4: Tokenization
def tokenize_function(example):
  return tokenizer(example["text"],truncation=True, padding="max_length")

tokenized_datasets = dataset.map(tokenize_function)

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

In [37]:
#step 5: Training
from transformers import Trainer , TrainingArguments
#add labels to the tokenized_dataset for causal language mdelling
def add_labels_to_dataset(examples):
  examples['labels']=examples['input_ids']
  return examples
tokenized_datasets=tokenized_datasets.map(add_labels_to_dataset,batched=True)
training_args=TrainingArguments(
    output_dir="./lora_model",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    logging_steps=10,
    save_steps=50
)
trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets
)
trainer.train()

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Step,Training Loss


TrainOutput(global_step=2, training_loss=5.463348388671875, metrics={'train_runtime': 11.3798, 'train_samples_per_second': 0.176, 'train_steps_per_second': 0.176, 'total_flos': 25451858755584.0, 'train_loss': 5.463348388671875, 'epoch': 2.0})

In [38]:
print(rag_query("What is Appointment Policy? "))

The Appointment Policy for MediCare Plus is as follows:

1. Appointments can be booked online via the website or mobile app.
2. Patients must book at least 30 minutes in advance for same-day consultations.
3. Appointments can be rescheduled or cancelled up to 15 minutes before the scheduled time.
4. A full refund is issued for cancellations made at least 1 hour before the appointment.
